# QAOA Vanilla - MaxCut on Weighted ER Graphs

This notebook:
- Generates 5 weighted Erdős–Rényi graphs
- Runs vanilla QAOA on each graph
- Collects expectation, variance, and optimal parameters

In [ ]:
from qaoa import QAOA, problems, mixers, initialstates
from qiskit_algorithms.optimizers import L_BFGS_B

from src.utils import generate_er_graphs

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [132]:
# Configuration
n_graphs = 5
n_nodes = 10
depth = 10   # QAOA layers (p)

In [133]:
# Generate graphs (already weighted from your function)
graphs = generate_er_graphs(n_graphs=n_graphs, n_nodes=n_nodes)

print(f"Generated {len(graphs)} graphs")

Generated 5 graphs


In [134]:
# Inspect one graph
name, G = list(graphs.items())[0]
print("Example graph:", name)
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Example graph: er_graph_0
Nodes: 10
Edges: 31


In [135]:
import itertools

def maxcut_bruteforce(G):
    n = G.number_of_nodes()
    best_cut = -float("inf")

    for bits in itertools.product([0, 1], repeat=n):
        cut = 0
        for u, v, w in G.edges(data="weight", default=1):
            if bits[u] != bits[v]:
                cut += w
        best_cut = max(best_cut, cut)

    return -best_cut

In [136]:
# Storage for results
results = []

In [ ]:
# Run QAOA on each graph
for graph_name, G in graphs.items():

    print(f"\nRunning QAOA on {graph_name}")

    # Initialize QAOA
    qaoa = QAOA(
        problem=problems.MaxCut(G),
        mixer=mixers.X(),
        initialstate=initialstates.Plus(),
        interpolate=True,
        optimizer=[L_BFGS_B, {
            "maxiter": 50,
            "ftol": 1e-9
        }]
    )

    # --- Step 1: Landscape (p=1 only, optional but useful)
    qaoa.sample_cost_landscape()

    # --- Step 2: Optimization
    qaoa.optimize(depth=depth)

    # --- Step 3: Extract results
    exp_val = qaoa.get_Exp(depth=depth)

    # Compute optimal
    cut_opt = maxcut_bruteforce(G)

    # Approximation Ratio
    ar = exp_val / cut_opt

    var_val = qaoa.get_Var(depth=depth)

    gamma = qaoa.get_gamma(depth=depth)
    beta = qaoa.get_beta(depth=depth)

    # Histogram (top states insight)
    hist = qaoa.hist

    # Store results
    results.append({
        "graph_name": graph_name,
        "n_nodes": G.number_of_nodes(),
        "n_edges": G.number_of_edges(),
        "depth_p": depth,
        "expectation": exp_val,
        "variance": var_val,
        "gamma": gamma,
        "beta": beta,
        "ar": ar
    })

    print("Expectation:", exp_val)
    print("Variance:", var_val)


Running QAOA on er_graph_0
2026-04-05 16:10:28 [info     ] Calculating energy landscape for depth p=1... file=qaoa.qaoa func=sample_cost_landscape


2026-04-05 16:10:28 [info     ] Executing sample_cost_landscape file=qaoa.qaoa func=sample_cost_landscape
2026-04-05 16:10:28 [info     ] circuits: 400                  file=qaoa.qaoa func=sample_cost_landscape
2026-04-05 16:10:28 [info     ] Done execute                   file=qaoa.qaoa func=sample_cost_landscape
2026-04-05 16:10:33 [info     ] Done measurement               file=qaoa.qaoa func=sample_cost_landscape
2026-04-05 16:10:33 [info     ] Calculating Energy landscape done file=qaoa.qaoa func=sample_cost_landscape
2026-04-05 16:10:35 [info     ] cost(depth 1 = -9.672998046874998 file=qaoa.qaoa func=optimize
2026-04-05 16:10:36 [info     ] cost(depth 2 = -9.719345703125 file=qaoa.qaoa func=optimize
2026-04-05 16:10:41 [info     ] cost(depth 3 = -10.336289062500002 file=qaoa.qaoa func=optimize
2026-04-05 16:10:46 [info     ] cost(depth 4 = -10.361982421874998 file=qaoa.qaoa func=optimize
2026-04-05 16:10:53 [info     ] cost(depth 5 = -10.469746093749997 file=qaoa.qaoa func=optim

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(results)

df

,graph_name,n_nodes,n_edges,depth_p,expectation,variance,gamma,beta,ar
0,er_graph_0,10,25,10,-8.352832,1.766334,"[0.6329161085031136, 0.6017611654791707, 0.5204173946670655, 0.5742348946680967, 0.5647117384628493, 0.5458666439909997, 0.5416060541828905, 0.4692966591306341, 0.41017495619351, 0.3826594347966258]","[0.41074346894659497, 0.2613280735964703, 0.20882095286274008, 0.24258025614524614, 0.23541635704539682, 0.24955279471628952, 0.24154470185770513, 0.13242337018157296, 0.15354145270130087, 0.32282209985004423]",0.854947


In [ ]:
# Sort by best expectation (MaxCut → usually minimize energy)
df_sorted = df.sort_values(by="expectation")

df_sorted

,graph_name,n_nodes,n_edges,depth_p,expectation,variance,gamma,beta,ar
0,er_graph_0,10,25,10,-8.352832,1.766334,"[0.6329161085031136, 0.6017611654791707, 0.5204173946670655, 0.5742348946680967, 0.5647117384628493, 0.5458666439909997, 0.5416060541828905, 0.4692966591306341, 0.41017495619351, 0.3826594347966258]","[0.41074346894659497, 0.2613280735964703, 0.20882095286274008, 0.24258025614524614, 0.23541635704539682, 0.24955279471628952, 0.24154470185770513, 0.13242337018157296, 0.15354145270130087, 0.32282209985004423]",0.854947


In [ ]:
# Inspect one result in detail
row = df_sorted.iloc[0]

print("Best graph:", row["graph_name"])
print("Gamma:", row["gamma"])
print("Beta:", row["beta"])
print("Expectation:", row["expectation"])
print("Variance:", row["variance"])

Best graph: er_graph_0
Gamma: [0.63291611 0.60176117 0.52041739 0.57423489 0.56471174 0.54586664
 0.54160605 0.46929666 0.41017496 0.38265943]
Beta: [0.41074347 0.26132807 0.20882095 0.24258026 0.23541636 0.24955279
 0.2415447  0.13242337 0.15354145 0.3228221 ]
Expectation: -8.352832031249996
Variance: 1.7663344937530507


In [ ]:
# Optional: analyze parameter patterns
import numpy as np

all_gamma = np.array(df["gamma"].tolist())
all_beta = np.array(df["beta"].tolist())

print("Mean gamma:", all_gamma.mean(axis=0))
print("Mean beta:", all_beta.mean(axis=0))

Mean gamma: [0.63291611 0.60176117 0.52041739 0.57423489 0.56471174 0.54586664
 0.54160605 0.46929666 0.41017496 0.38265943]
Mean beta: [0.41074347 0.26132807 0.20882095 0.24258026 0.23541636 0.24955279
 0.2415447  0.13242337 0.15354145 0.3228221 ]


In [ ]:
# Done
print("QAOA experiment complete ✅")

QAOA experiment complete ✅
